# Imports and Data Read

In [ ]:
import pandas as pd
import numpy as np
import re
import ast
import emoji
from dotenv import load_dotenv
import os
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

In [35]:
df = pd.read_csv("/Users/ziadsamer/Documents/who-will-viral/data/youtube/cleaned_dataset.csv", keep_default_na=False)

In [36]:
df.head()

,video_id,title,publishedAt,channelId,channelTitle,categoryId,tags,view_count,likes,comment_count,...,projection,embeddable,madeForKids,chapter_count,playability_status,supports_miniplayer,card_count,is_verified,has_paid_promotion,comments_disabled
0,G4M_621v1As,college farewell video#trending #viralshorts,2025-04-12T02:06:42Z,UC7k_7IF3s3TY-cowwxs_yWw,Vk_07_rider,22,"['college farewell video', 'DDU farewell video...",18.650077,14.258386,7.367709,...,rectangular,True,False,0,OK,True,1,False,False,False
1,z2voqo_Jhx4,Busking in Manchester 🇬🇧 #blindfaith #guitar #...,2025-04-06T11:35:53Z,UCM_9JHB3xXPSzJfTkD86AtA,Leonardo Patrick,10,[],18.366689,13.738122,8.674197,...,rectangular,True,False,0,OK,True,1,False,False,False
2,jHIt9oHFLsw,This is what happens when you play Star Wars i...,2025-04-06T10:07:12Z,UC4YyKY5o60Kckk-GXOnhT2g,Violin Phonix,22,[],17.941771,13.880749,8.324821,...,rectangular,True,False,0,OK,True,1,True,False,False
3,gwRqLbWqKlM,LISA - FUTW (YouTube Music Nights Special Stag...,2025-03-19T03:29:33Z,UC6-BgjsBa5R3PZQ_kZ8hKPg,LLOUD Official,10,"['Blackpink', 'Lisa', 'Music', 'Fashion', 'K-P...",16.493511,13.029847,9.923143,...,rectangular,True,False,0,OK,True,1,True,False,False
4,prpRoyrutcE,Uljhi hai yeh kis jaal me tu…. Bengaluru ❤️,2025-04-14T10:17:58Z,UCiFXnvi8ESDukE25ol2foWQ,Mr.KiranJ,10,[],17.081459,12.725664,7.755339,...,rectangular,True,False,0,OK,True,1,True,False,False


In [37]:
df.columns

Index(['video_id', 'title', 'publishedAt', 'channelId', 'channelTitle',
       'categoryId', 'tags', 'view_count', 'likes', 'comment_count',
       'description', 'is_trending', 'defaultLanguage', 'duration',
       'dimension', 'definition', 'caption', 'licensedContent', 'projection',
       'embeddable', 'madeForKids', 'chapter_count', 'playability_status',
       'supports_miniplayer', 'card_count', 'is_verified',
       'has_paid_promotion', 'comments_disabled'],
      dtype='str')

# Feature Engineering

## Feature Interactions

In [38]:
df["like_to_view_ratio"] = df["likes"] / df["view_count"]

In [39]:
df["comment_to_view_ratio"] = df["comment_count"] / df["view_count"]

## Dates and Str features

In [40]:
df["tag_count"] = df["tags"].apply(lambda x: len(x) if isinstance(x, list) else len(x.split(",")) if isinstance(x, str) else 0)

In [41]:
df["title_length"] = df["title"].apply(lambda x: len(x))

In [42]:
df["description_length"] = df["description"].apply(lambda x: len(x) if isinstance(x, str) else 0)

In [43]:
df["title_has_caps_ratio"] = df["title"].apply(lambda x: sum(1 for c in x if c.isupper()) / len(x))

In [44]:
df["publishedAt"]

0         2025-04-12T02:06:42Z
1         2025-04-06T11:35:53Z
2         2025-04-06T10:07:12Z
3         2025-03-19T03:29:33Z
4         2025-04-14T10:17:58Z
                  ...         
180622    2026-04-06T13:56:10Z
180623    2026-04-06T13:54:10Z
180624    2026-04-06T13:48:23Z
180625    2026-04-06T13:45:02Z
180626    2026-04-06T13:43:23Z
Name: publishedAt, Length: 180627, dtype: str

In [45]:
df["publish_hour"] = pd.to_datetime(df["publishedAt"]).dt.hour
df["publish_dayofweek"] = pd.to_datetime(df["publishedAt"]).dt.dayofweek

In [46]:
def get_duration_seconds(x):
    if not isinstance(x, str):
        return 0
    
    match = re.match(r"^P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?)?$", x)
    if not match:
        return 0
    
    days    = int(match.group(1) or 0)
    hours   = int(match.group(2) or 0)
    minutes = int(match.group(3) or 0)
    seconds = int(match.group(4) or 0)
    
    return (days * 86400) + (hours * 3600) + (minutes * 60) + seconds

df["duration_seconds"] = df["duration"].apply(get_duration_seconds)

In [47]:
def count_emojis(text):
    emoji_list = [ch for ch in emoji.emoji_list(text)]
    return len(emoji_list)

In [50]:
df["description_emojis"] = df["description"].apply(count_emojis)
df["title_emojis"] = df["title"].apply(count_emojis)

## Embeddings Features

In [60]:
def parse_tags(val):
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except:
        return []

df["tags"] = df["tags"].apply(parse_tags)
df["tags_joined"] = df["tags"].apply(lambda tags: " ".join(tags))

0    college farewell video DDU farewell video fare...
1                                                     
2                                                     
3    Blackpink Lisa Music Fashion K-Pop kpop LLoud ...
4                                                     
Name: tags_joined, dtype: str


In [69]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [70]:
model = SentenceTransformer("all-MiniLM-L6-v2", token=hf_token)
tags_for_embedding = df["tags_joined"].replace("", "no tags")
embeddings = model.encode(tags_for_embedding.tolist(), show_progress_bar=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16119.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 5645/5645 [11:04<00:00,  8.49it/s]   


In [ ]:
pca = PCA(n_components=50)
embeddings_reduced = pca.fit_transform(embeddings)
print(pca.explained_variance_ratio_.sum())  
print(pca.explained_variance_ratio_) 

In [73]:
embeddings_df = pd.DataFrame(embeddings_reduced, columns=[f"tag_emb_{i}" for i in range(embeddings_reduced.shape[1])])
df = pd.concat([df.reset_index(drop=True), embeddings_df], axis=1)

,video_id,title,publishedAt,channelId,channelTitle,categoryId,tags,view_count,likes,comment_count,...,tag_emb_40,tag_emb_41,tag_emb_42,tag_emb_43,tag_emb_44,tag_emb_45,tag_emb_46,tag_emb_47,tag_emb_48,tag_emb_49
0,G4M_621v1As,college farewell video#trending #viralshorts,2025-04-12T02:06:42Z,UC7k_7IF3s3TY-cowwxs_yWw,Vk_07_rider,22,"[college farewell video, DDU farewell video, f...",18.650077,14.258386,7.367709,...,-0.035558,-0.020191,0.061964,0.106967,-0.033296,-0.148294,-0.032410,0.001595,0.059156,-0.005476
1,z2voqo_Jhx4,Busking in Manchester 🇬🇧 #blindfaith #guitar #...,2025-04-06T11:35:53Z,UCM_9JHB3xXPSzJfTkD86AtA,Leonardo Patrick,10,[],18.366689,13.738122,8.674197,...,0.000429,-0.000211,-0.000309,0.000168,0.000154,0.000054,-0.000470,-0.000081,-0.000061,-0.000286
2,jHIt9oHFLsw,This is what happens when you play Star Wars i...,2025-04-06T10:07:12Z,UC4YyKY5o60Kckk-GXOnhT2g,Violin Phonix,22,[],17.941771,13.880749,8.324821,...,0.000429,-0.000211,-0.000309,0.000168,0.000154,0.000054,-0.000470,-0.000081,-0.000061,-0.000286
3,gwRqLbWqKlM,LISA - FUTW (YouTube Music Nights Special Stag...,2025-03-19T03:29:33Z,UC6-BgjsBa5R3PZQ_kZ8hKPg,LLOUD Official,10,"[Blackpink, Lisa, Music, Fashion, K-Pop, kpop,...",16.493511,13.029847,9.923143,...,0.097198,-0.064388,-0.057791,0.047045,0.061149,0.049138,-0.078962,-0.031099,-0.056352,-0.069381
4,prpRoyrutcE,Uljhi hai yeh kis jaal me tu…. Bengaluru ❤️,2025-04-14T10:17:58Z,UCiFXnvi8ESDukE25ol2foWQ,Mr.KiranJ,10,[],17.081459,12.725664,7.755339,...,0.000429,-0.000211,-0.000309,0.000168,0.000154,0.000054,-0.000470,-0.000081,-0.000061,-0.000286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180622,NDlIoXE-Qow,electric boy #arun #experiment #trending #scie...,2026-04-06T13:56:10Z,UCLhWspuWMU_5HrmDa5oFoKQ,Tec mastar,28,[],6.777647,2.833213,0.000000,...,0.000429,-0.000211,-0.000309,0.000168,0.000154,0.000054,-0.000470,-0.000081,-0.000061,-0.000286
180623,xMrHBO5Gtr4,Chamach Science Experiment #shorts #experiment...,2026-04-06T13:54:10Z,UCaosO2NO5X5aQ8ZJSHLQYHQ,Shubham K Experiment,28,"['science experiment with chamach', 'simple sc...",7.377759,2.484907,0.000000,...,-0.088179,-0.100553,0.013869,-0.082732,0.140938,-0.128951,0.027207,-0.056198,-0.064932,0.001402
180624,Rf4D-d4pcfM,मिसाइल कैसे काम करती हैं? missile kaise kam ka...,2026-04-06T13:48:23Z,UCx9L7nOME1GI8ljGp4MoGuw,KKB GAMER,22,[],7.026427,3.663562,0.693147,...,0.000429,-0.000211,-0.000309,0.000168,0.000154,0.000054,-0.000470,-0.000081,-0.000061,-0.000286
180625,B00XXVjk6Lk,Why Does Snow Look White? #facts #kidseducatio...,2026-04-06T13:45:02Z,UCXXbYBfrrCl-4ACwNxXqz6Q,Rhyme Time,27,"['kidslearning', 'scienceforkids', 'nature', '...",7.263330,2.772589,0.000000,...,-0.047726,0.100043,-0.085869,-0.166551,0.133809,-0.036055,0.032443,0.009027,0.027120,-0.035237


### Description

In [76]:
model = SentenceTransformer("all-MiniLM-L6-v2", token=hf_token)
description_for_embedding = df["description"].replace("", "no description")
embeddings = model.encode(description_for_embedding.tolist(), show_progress_bar=True, batch_size=256)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11091.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 706/706 [04:46<00:00,  2.47it/s]


In [78]:
pca = PCA(n_components=25)
embeddings_reduced = pca.fit_transform(embeddings)
print(pca.explained_variance_ratio_.sum())  
print(pca.explained_variance_ratio_) 

0.4774933
[0.18903817 0.03655031 0.02557514 0.01956713 0.01825797 0.01608934
 0.01528703 0.01345869 0.01260801 0.01116739 0.01080527 0.01052562
 0.00962847 0.00932768 0.00854209 0.00808513 0.00802983 0.00770661
 0.0075008  0.00715692 0.0069089  0.00671399 0.00650959 0.00626978
 0.00618337]


In [79]:
embeddings_df = pd.DataFrame(embeddings_reduced, columns=[f"description_emb_{i}" for i in range(embeddings_reduced.shape[1])])
df = pd.concat([df.reset_index(drop=True), embeddings_df], axis=1)

### Title

In [98]:
model = SentenceTransformer("all-MiniLM-L6-v2", token=hf_token)
title_for_embedding = df["title"].replace("", "no text")
embeddings = model.encode(title_for_embedding.tolist(), show_progress_bar=True, batch_size=256)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8650.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 706/706 [02:55<00:00,  4.01it/s]


In [99]:
pca = PCA(n_components=25)
embeddings_reduced = pca.fit_transform(embeddings)
print(pca.explained_variance_ratio_.sum())  
print(pca.explained_variance_ratio_) 

0.36500794
[0.04697935 0.03515688 0.02641294 0.01965372 0.01834958 0.01774918
 0.01665124 0.01478205 0.0137953  0.01307299 0.01287447 0.01233836
 0.01105979 0.01069745 0.01032169 0.01011367 0.0096083  0.00894795
 0.00867045 0.00840564 0.00826064 0.00814665 0.00781711 0.00772198
 0.00742061]


In [100]:
embeddings_df = pd.DataFrame(embeddings_reduced, columns=[f"title_emb_{i}" for i in range(embeddings_reduced.shape[1])])
df = pd.concat([df.reset_index(drop=True), embeddings_df], axis=1)

# Baseline Model for checks

In [101]:
df_numeric = df.select_dtypes(include=["int64", "float64", "int32", "float32"])
df_numeric.head()

,categoryId,view_count,likes,comment_count,is_trending,chapter_count,card_count,like_to_view_ratio,comment_to_view_ratio,tag_count,...,title_emb_15,title_emb_16,title_emb_17,title_emb_18,title_emb_19,title_emb_20,title_emb_21,title_emb_22,title_emb_23,title_emb_24
0,22,18.650077,14.258386,7.367709,1,0,1,0.764522,0.395050,6,...,0.164791,0.064555,-0.057024,0.103280,-0.071986,0.049581,-0.061173,-0.045241,-0.127490,0.076514
1,10,18.366689,13.738122,8.674197,1,0,1,0.747991,0.472279,1,...,-0.015482,-0.209932,-0.260757,-0.207158,-0.038436,-0.137065,0.048197,-0.036557,0.038056,0.077784
2,22,17.941771,13.880749,8.324821,1,0,1,0.773655,0.463991,1,...,-0.065034,-0.091152,0.034013,-0.130892,0.014330,0.017098,-0.175022,-0.072930,-0.197116,-0.035836
3,10,16.493511,13.029847,9.923143,1,0,1,0.789998,0.601639,50,...,-0.079036,-0.001199,-0.087550,-0.009422,0.049827,0.040932,-0.079335,0.127770,0.044866,-0.066416
4,10,17.081459,12.725664,7.755339,1,0,1,0.744999,0.454021,1,...,-0.060052,-0.070425,0.007224,-0.097888,-0.001817,-0.032237,-0.062615,0.015606,0.057677,-0.009284


In [102]:
df_numeric['is_trending'].value_counts()

is_trending
0    148559
1     32068
Name: count, dtype: int64

In [103]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

In [104]:
X = df_numeric.drop("is_trending", axis=1).values
y = df_numeric["is_trending"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = f1_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

print(classification_report(y_test, y_pred))

Accuracy: 0.9202950273614086
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     29819
           1       0.92      0.92      0.92      6307

    accuracy                           0.97     36126
   macro avg       0.95      0.95      0.95     36126
weighted avg       0.97      0.97      0.97     36126



In [ ]:
# were removed?
# is_region_restricted
# blocked_regions_count
# aloowed_regions_count

# has_emojis
